In [ ]:
import json
import re
from pathlib import Path

WHEN_DIR  = Path("../outputs/when")
PATCH_DIR = Path("../outputs/patch")

patch_files = sorted(PATCH_DIR.glob("*.json"))
records     = [json.loads(p.read_text()) for p in sorted(WHEN_DIR.glob("*.json"))]
with_delta  = [r for r in records if r.get("delta_days") is not None]

print(f"Total patch files:         {len(patch_files)}")
print(f"Total when records:        {len(records)}")
print(f"  Records with delta_days: {len(with_delta)}")
print(f"  Records without (no date): {len(records) - len(with_delta)}")

In [ ]:
_GITHUB_RESOLVABLE_RE = re.compile(
    r"github\.com/[^/]+/[^/]+/(?:commit/[0-9a-f]{5,}|pull/\d+|releases/tag/[^/?#]+)"
)
_ISSUE_RE    = re.compile(r"github\.com/[^/]+/[^/]+/issues/")
_BLAME_RE    = re.compile(r"github\.com/[^/]+/[^/]+/blame/")
_REPO_ROOT_RE = re.compile(r"github\.com/[^/]+/[^/?#]+/?(?:[?#].*)?$")

when_ids = {p.stem for p in WHEN_DIR.glob("*.json")}

# --- Gap 1: patch files that produced no when record ---
no_github_link   = 0
unresolvable     = {"issues": 0, "blame": 0, "repo_root": 0, "other_github": 0}

for p in patch_files:
    if p.stem in when_ids:
        continue
    urls = [x.get("url", "") for x in json.loads(p.read_text()).get("patches", [])]
    if not any("github.com" in u for u in urls):
        no_github_link += 1
        continue
    for u in urls:
        if _ISSUE_RE.search(u):
            unresolvable["issues"] += 1
        elif _BLAME_RE.search(u):
            unresolvable["blame"] += 1
        elif not _GITHUB_RESOLVABLE_RE.search(u) and _REPO_ROOT_RE.search(u):
            unresolvable["repo_root"] += 1
        elif not _GITHUB_RESOLVABLE_RE.search(u) and "github.com" in u:
            unresolvable["other_github"] += 1

missing_when = len(patch_files) - len(when_ids)
unresolvable_total = sum(unresolvable.values())

print("=== Patch files with no when record ===")
print(f"  Total missing: {missing_when}")
print(f"  Reason 1 — No GitHub link at all (non-GitHub URLs tagged 'patch'): {no_github_link}")
print(f"  Reason 2 — GitHub link present but type has no date API:           {missing_when - no_github_link}")
print(f"    /issues/  (issue tracker, no fix-date endpoint): {unresolvable['issues']}")
print(f"    /blame/   (file blame view, no date endpoint):   {unresolvable['blame']}")
print(f"    repo root (no specific commit/release):          {unresolvable['repo_root']}")
print(f"    other     (wiki, gist, advisory, etc.):         {unresolvable['other_github']}")

# --- Gap 2: when records with no delta_days ---
no_delta         = [r for r in records if r.get("delta_days") is None]
no_patch_date    = [r for r in no_delta if     r.get("cve_published_date") and not r.get("soonest_patched_date")]
no_cve_date      = [r for r in no_delta if not r.get("cve_published_date") and     r.get("soonest_patched_date")]
both_missing     = [r for r in no_delta if not r.get("cve_published_date") and not r.get("soonest_patched_date")]

print()
print("=== When records with no delta_days ===")
print(f"  Total without date: {len(no_delta)}")
print(f"  Reason — GitHub API returned no date for any patch URL")
print(f"    (repo deleted, private, or SHA not found):  {len(no_patch_date)}")
print(f"  Reason — CVE has no datePublished in metadata: {len(no_cve_date)}")
print(f"  Reason — Both dates missing:                    {len(both_missing)}")
print()
print("  Sample CVEs where GitHub API returned no patch date:")
for r in no_patch_date[:5]:
    patches = r.get("patches", [])
    url = patches[0].get("url", "?") if patches else "?"
    print(f"    {r['id']:20}  url={url}")

In [ ]:
cve_first   = [r for r in with_delta if r["cve_first"] is True]
patch_first = [r for r in with_delta if r["cve_first"] is False]

total = len(with_delta)
print(f"CVE published first (patch came later):  {len(cve_first):>5}  ({len(cve_first)/total:.1%})")
print(f"Patch existed first (CVE came later):    {len(patch_first):>5}  ({len(patch_first)/total:.1%})")

In [ ]:
import statistics

deltas = [r["delta_days"] for r in with_delta]
deltas_sorted = sorted(deltas)
n = len(deltas)

def percentile(data, p):
    k = (len(data) - 1) * p / 100
    lo, hi = int(k), min(int(k) + 1, len(data) - 1)
    return data[lo] + (data[hi] - data[lo]) * (k - lo)

print(f"Count:   {n}")
print(f"Min:     {min(deltas):.2f} days")
print(f"P25:     {percentile(deltas_sorted, 25):.2f} days")
print(f"Median:  {statistics.median(deltas):.2f} days")
print(f"Mean:    {statistics.mean(deltas):.2f} days")
print(f"P75:     {percentile(deltas_sorted, 75):.2f} days")
print(f"Max:     {max(deltas):.2f} days")
print(f"Stdev:   {statistics.stdev(deltas):.2f} days")

In [ ]:
buckets = [
    ("patch > 1 year before CVE",  lambda d: d < -365),
    ("patch 6–12 months before",   lambda d: -365 <= d < -180),
    ("patch 1–6 months before",    lambda d: -180 <= d < -30),
    ("patch 0–30 days before",     lambda d: -30  <= d < 0),
    ("same day (within 1 day)",    lambda d: -1   <= d <= 1),
    ("patch 0–30 days after CVE",  lambda d: 1    <  d <= 30),
    ("patch 1–6 months after",     lambda d: 30   <  d <= 180),
    ("patch > 6 months after CVE", lambda d: d > 180),
]

print(f"{'Bucket':<35} {'Count':>6}  {'%':>6}")
print("-" * 52)
for label, fn in buckets:
    count = sum(1 for d in deltas if fn(d))
    print(f"{label:<35} {count:>6}  {count/n:>5.1%}")

## Extremes — Fastest and Slowest Patches

In [ ]:
ranked = sorted(with_delta, key=lambda r: r["delta_days"])

def _show(r):
    return (f"{r['id']:20}  delta={r['delta_days']:>10.2f}d  "
            f"cve={r['cve_published_date'][:10]}  patch={r['soonest_patched_date'][:10]}")

print("=== Patch existed LONGEST before CVE (most negative delta) ===")
for r in ranked[:5]:
    print(" ", _show(r))

print("\n=== Patch came LONGEST after CVE (most positive delta) ===")
for r in ranked[-5:][::-1]:
    print(" ", _show(r))

In [ ]:
from collections import defaultdict

# Diagnostic: check how many with_delta records have a cve_published_date
missing_date = [r for r in with_delta if not r.get("cve_published_date")]
print(f"with_delta records missing cve_published_date: {len(missing_date)} / {len(with_delta)}")
sample = [r.get("cve_published_date") for r in with_delta[:5]]
print(f"Sample cve_published_date values: {sample}")

year_deltas = defaultdict(list)
for r in with_delta:
    raw = r.get("cve_published_date")
    if not raw:
        continue
    year = str(raw)[:4]
    year_deltas[year].append(r["delta_days"])

print(f"\nDistinct years found: {sorted(year_deltas)}\n")
print(f"{'Year':<6} {'Count':>6}  {'Mean (days)':>12}  {'Median (days)':>14}  {'Stdev (days)':>13}")
print("-" * 58)
for year in sorted(year_deltas):
    vals = sorted(year_deltas[year])
    n_y = len(vals)
    mean_y = statistics.mean(vals)
    median_y = statistics.median(vals)
    stdev_y = statistics.stdev(vals) if n_y > 1 else 0.0
    print(f"{year:<6} {n_y:>6}  {mean_y:>12.2f}  {median_y:>14.2f}  {stdev_y:>13.2f}")